In [56]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
from collections import defaultdict
import warnings

In [57]:
warnings.filterwarnings("ignore", category=UserWarning)

In [58]:
menu = [
    {"name": "Biryani", "category": "main", "price": 220},
    {"name": "Paneer Tikka", "category": "starter", "price": 180},
    {"name": "Butter Naan", "category": "bread", "price": 40},
    {"name": "Raita", "category": "side", "price": 50},
    {"name": "Gulab Jamun", "category": "dessert", "price": 90},
    {"name": "Cold Drink", "category": "drink", "price": 60},
    {"name": "Lassi", "category": "drink", "price": 70},
    {"name": "Chicken Kebab", "category": "starter", "price": 210}
]

In [59]:
orders = [
    {"items": ["Biryani", "Raita", "Cold Drink"]},
    {"items": ["Paneer Tikka", "Butter Naan", "Lassi"]},
    {"items": ["Biryani", "Gulab Jamun"]},
    {"items": ["Chicken Kebab", "Cold Drink"]},
    {"items": ["Biryani", "Raita"]},
    {"items": ["Paneer Tikka", "Lassi"]}
]

In [60]:
# Finding which items are frequently bought together
co_occurrence = defaultdict(lambda: defaultdict(int))
for order in orders:
    items = order['items']
    for i in items:
        for j in items:
            if i != j: co_occurrence[i][j] += 1

In [61]:
candidate_map = {item: sorted(neighbors, key=neighbors.get, reverse=True) 
                 for item, neighbors in co_occurrence.items()}

In [62]:
training_rows = []
all_names = [m['name'] for m in menu]
for order in orders:
    items = order['items']
    if len(items) < 2: continue
    target, cart = items[-1], items[:-1]
    
    # Positive: What user actually added
    training_rows.append({
        "cart_size": len(cart),
        "price": next(m['price'] for m in menu if m['name'] == target),
        "is_drink": 1 if next(m['category'] for m in menu if m['name'] == target) == 'drink' else 0,
        "label": 1
    })
    # Negative: Random item they didn't add
    neg = np.random.choice([n for n in all_names if n not in items])
    training_rows.append({
        "cart_size": len(cart),
        "price": next(m['price'] for m in menu if m['name'] == neg),
        "is_drink": 1 if next(m['category'] for m in menu if m['name'] == neg) == 'drink' else 0,
        "label": 0
    })

In [63]:
df = pd.DataFrame(training_rows)
print(df)

    cart_size  price  is_drink  label
0           2     60         1      1
1           2     90         0      0
2           2     70         1      1
3           2     60         1      0
4           1     90         0      1
5           1     70         1      0
6           1     60         1      1
7           1     50         0      0
8           1     50         0      1
9           1    180         0      0
10          1     70         1      1
11          1     60         1      0


In [64]:
model_lgb = lgb.LGBMClassifier(n_estimators=50, min_child_samples=1, min_data_in_bin=1)
model_lgb.fit(df.drop('label', axis=1), df['label'])

[LightGBM] [Info] Number of positive: 6, number of negative: 6
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11
[LightGBM] [Info] Number of data points in the train set: 12, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

LGBMClassifier(min_child_samples=1, min_data_in_bin=1, n_estimators=50)

In [65]:
class RecommendationService:
    def __init__(self, lgb_model, candidates, menu_data):
        self.model = lgb_model
        self.candidates = candidates
        self.item_meta = {m['name']: m for m in menu_data}
        # Save feature names to ensure consistency
        self.feature_cols = ["cart_size", "price", "is_drink"]

    def get_recommendations(self, cart_items, top_n=3):
        last_item = cart_items[-1]
        if last_item not in self.candidates: return []
        
        cands = self.candidates[last_item]
        features_list = []
        for name in cands:
            item = self.item_meta[name]
            features_list.append({
                "cart_size": len(cart_items),
                "price": item['price'],
                "is_drink": 1 if item['category'] == 'drink' else 0
            })
        
     
        features_df = pd.DataFrame(features_list)[self.feature_cols]
        
        scores = self.model.predict_proba(features_df)[:, 1]
        ranked = sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)
        return [r[0] for r in ranked if r[0] not in cart_items][:top_n]


In [66]:
recommender = RecommendationService(model_lgb, candidate_map, menu)

In [67]:
# Assuming you have initialized 'recommender' from the previous code

def run_tests(service):
    print("🚀 Starting Complex Model Validation...\n")
    
    # --- TEST 1: Cold Start (Unknown Item) ---
    print("Test 1 [Cold Start]: Item 'Sushi' (not in menu)")
    # Should handle gracefully without crashing
    res_1 = service.get_recommendations(['Sushi']) 
    print(f"Result: {res_1} | Pass: {res_1 == []}\n")

    # --- TEST 2: Deduplication (Existing Item) ---
    print("Test 2 [Deduplication]: Cart already has Raita")
    # Should not suggest Raita even if Biryani usually triggers it
    res_2 = service.get_recommendations(['Biryani', 'Raita'])
    print(f"Result: {res_2} | Pass: {'Raita' not in res_2}\n")

    # --- TEST 3: Category Logic (Main -> Drink/Side) ---
    print("Test 3 [Synergy]: High-price Main in cart")
    res_3 = service.get_recommendations(['Biryani'])
    # Check if a drink or side is at the top
    top_item_cat = service.item_meta[res_3[0]]['category']
    print(f"Top Rec: {res_3[0]} ({top_item_cat}) | Pass: {top_item_cat in ['side', 'drink']}\n")

    # --- TEST 4: Performance (Large Input) ---
    import time
    print("Test 4 [Performance]: 10-item cart stress test")
    large_cart = ['Biryani'] * 10
    start = time.time()
    service.get_recommendations(large_cart)
    duration = (time.time() - start) * 1000
    print(f"Latency: {duration:.2f}ms | Pass: {duration < 100}\n")

run_tests(recommender)

🚀 Starting Complex Model Validation...

Test 1 [Cold Start]: Item 'Sushi' (not in menu)
Result: [] | Pass: True

Test 2 [Deduplication]: Cart already has Raita
Result: ['Cold Drink'] | Pass: True

Test 3 [Synergy]: High-price Main in cart
Top Rec: Gulab Jamun (dessert) | Pass: False

Test 4 [Performance]: 10-item cart stress test
Latency: 1.57ms | Pass: True



In [68]:
import time
import pandas as pd

def manual_test_with_latency(service):
    print("Available items:", ", ".join(service.item_meta.keys()))
    print("Type 'exit' to stop.")
    
    while True:
        user_input = input("\nEnter items in cart (comma separated): ")
        if user_input.lower() == 'exit': break
 
        cart = [i.strip() for i in user_input.split(',')]
   
        valid_cart = [i for i in cart if i in service.item_meta]
        if not valid_cart:
            print("None of those items are in the menu. Try again.")
            continue

        start_time = time.perf_counter() # High-precision timer
        
        try:
            recommendations = service.get_recommendations(valid_cart)
            end_time = time.perf_counter()

            latency_ms = (end_time - start_time) * 1000

            print(f"\nModel Latency: {latency_ms:.4f} ms")
            print(f"Cart Processed: {valid_cart}")
            if recommendations:
                print(f"Top Recommendations: {recommendations}")
            else:
                print("No specific recommendations found for this combination.")

            if latency_ms < 200:
                print("Performance: Within target (<200ms)")
            else:
                print("Performance: Slow (Exceeds 200ms)")

        except Exception as e:
            print(f"Error during inference: {e}")

manual_test_with_latency(recommender)

Available items: Biryani, Paneer Tikka, Butter Naan, Raita, Gulab Jamun, Cold Drink, Lassi, Chicken Kebab
Type 'exit' to stop.



Enter items in cart (comma separated):  Cold Drink, Lassi



Model Latency: 3.7906 ms
Cart Processed: ['Cold Drink', 'Lassi']
Top Recommendations: ['Butter Naan', 'Paneer Tikka']
Performance: Within target (<200ms)



Enter items in cart (comma separated):  Paneer Tikka, Butter Naan



Model Latency: 3.7811 ms
Cart Processed: ['Paneer Tikka', 'Butter Naan']
Top Recommendations: ['Lassi']
Performance: Within target (<200ms)



Enter items in cart (comma separated):  exit


In [70]:
import time
import numpy as np

def stress_test_model(service, iterations=10000):
    all_items = list(service.item_meta.keys())
    latencies = []

    print(f"imulating {iterations} real-world requests...")
    
    for _ in range(iterations):
        # Generate a random cart of 1-5 items
        cart_size = np.random.randint(1, 6)
        random_cart = np.random.choice(all_items, size=cart_size, replace=False).tolist()
        
        start = time.perf_counter()
        service.get_recommendations(random_cart)
        latencies.append((time.perf_counter() - start) * 1000)

    print(f"Stress Test Complete")
    print(f" Avg Latency: {np.mean(latencies):.2f} ms")
    print(f" P99 Latency (Worst Case): {np.percentile(latencies, 99):.2f} ms")

stress_test_model(recommender)

imulating 10000 real-world requests...
Stress Test Complete
 Avg Latency: 2.31 ms
 P99 Latency (Worst Case): 2.68 ms


In [71]:
def stress_test_model(service, iterations=10000):
    all_items = list(service.item_meta.keys())
    latencies = []

    print(f"Simulating {iterations} real-world requests...")
    
    # Progress indicator
    checkpoint = iterations // 10
    
    for i in range(iterations):
        cart_size = np.random.randint(1, 4)
        random_cart = np.random.choice(all_items, size=cart_size, replace=False).tolist()
        
        start = time.perf_counter()
        service.get_recommendations(random_cart)
        latencies.append((time.perf_counter() - start) * 1000)
        
        if (i + 1) % checkpoint == 0:
            print(f"{((i + 1) / iterations) * 100:.0f}% Complete...")

    print("\n---Final Performance Report ---")
    print(f"Total Requests: {iterations}")
    print(f" Avg Latency:   {np.mean(latencies):.2f} ms")
    print(f" P99 (Worst):    {np.percentile(latencies, 99):.2f} ms")
    print(f" Status:        {'PASS' if np.mean(latencies) < 10 else 'OPTIMIZE NEEDED'}")


recommender = RecommendationService(model_lgb, candidate_map, menu)
stress_test_model(recommender)

Simulating 10000 real-world requests...
10% Complete...
20% Complete...
30% Complete...
40% Complete...
50% Complete...
60% Complete...
70% Complete...
80% Complete...
90% Complete...
100% Complete...

---Final Performance Report ---
Total Requests: 10000
 Avg Latency:   2.42 ms
 P99 (Worst):    2.68 ms
 Status:        PASS
